<a href="https://colab.research.google.com/github/Rindhya533/CLSA_Assessment_1/blob/main/CLSA_Assessment_Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CLSA ECG Practical Assessment – Part One -
Name: Morreddigari Rindhya Reddy

Questions

1.   Using R/Python, find out which stocks have missing price data
2.   Using R/Python, find out which price column (p1, p2, p3, or p4) corresponds to which price (open, high, low or close)
1.   Using R/Python, calculate the annualized daily volatility of each stock for which data is present


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving niftyprices.csv to niftyprices.csv


In [ ]:
df = pd.read_csv("niftyprices.csv")

In [ ]:
df.head()

,ticker,date,p1,p2,p3,p4
0,UPLL IN Equity,2021-11-26,701.00,726.00,703.80,726.00
1,HMCL IN Equity,2021-11-03,2582.90,2639.64,2584.32,2640.53
2,TTAN IN Equity,2021-07-07,1713.44,1763.22,1723.12,1779.13
3,DIVI IN Equity,2021-08-13,4838.52,4894.74,4928.50,4954.24
4,TATACONS IN Equity,2021-04-05,635.52,647.25,641.23,648.19


In [ ]:
df.columns

Index(['ticker', 'date', 'p1', 'p2', 'p3', 'p4'], dtype='object')

The dataset was checked for null values across all price columns.

In [ ]:
df.isnull().sum()

,0
ticker,0
date,0
p1,496
p2,496
p3,496
p4,496


In [ ]:
missing_rows = df[df.isnull().any(axis=1)]

missing_rows['ticker'].unique()

array(['COAL IN Equity', 'JSTL IN Equity'], dtype=object)

The following stocks contain missing values:

- COAL IN Equity
- JSTL IN Equity

In [ ]:
(df[['p1','p2','p3','p4']].idxmax(axis=1)).value_counts()

/tmp/ipykernel_7073/3562941605.py:1: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  (df[['p1','p2','p3','p4']].idxmax(axis=1)).value_counts()


,count
p4,10800
p2,1104


p4 appeared most frequently as the maximum value and is identified as the High price.

In [ ]:
(df[['p1','p2','p3','p4']].idxmin(axis=1)).value_counts()

/tmp/ipykernel_7073/2316521562.py:1: FutureWarning: The behavior of DataFrame.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  (df[['p1','p2','p3','p4']].idxmin(axis=1)).value_counts()


,count
p1,11904


p1 appeared most frequently as the minimum value and is identified as the Low price.

In [ ]:
df['date'] = pd.to_datetime(df['date'])

df = df.sort_values(['ticker', 'date'])

In [ ]:
df['p2_prev'] = df.groupby('ticker')['p2'].shift(1)
df['p3_prev'] = df.groupby('ticker')['p3'].shift(1)

In [ ]:
diff1 = (df['p2'] - df['p3_prev']).abs().mean()
diff2 = (df['p3'] - df['p2_prev']).abs().mean()

print("p2 vs previous p3:", diff1)
print("p3 vs previous p2:", diff2)

p2 vs previous p3: 16.08666413630229
p3 vs previous p2: 52.863383097165986


Since p2 showed smaller deviation from the previous day's p3 values, p2 is identified as the Open price and p3 as the Close price.

| Actual Price | Column |
|---|---|
| Open | p2 |
| High | p4 |
| Low | p1 |
| Close | p3 |

In [ ]:
df['return']=df.groupby('ticker')['p3'].transform(
    lambda x: np.log(x/x.shift(1))
)


Daily logarithmic returns were calculated using the Close price. Annualized volatility was computed using:

In [ ]:
volatility=df.groupby('ticker')['return'].std()*np.sqrt(252)

In [ ]:
volatility = volatility.reset_index()
volatility.columns = ['ticker', 'annualized_volatility']
volatility

,ticker,annualized_volatility
0,ADSEZ IN Equity,0.409058
1,APNT IN Equity,0.268714
2,AXSB IN Equity,0.301975
3,BAF IN Equity,0.329862
4,BHARTI IN Equity,0.278360
5,BJAUT IN Equity,0.244243
6,BJFIN IN Equity,0.337865
7,BPCL IN Equity,0.259885
8,BRIT IN Equity,0.179495
9,CIPLA IN Equity,0.246501


In [ ]:
volatility.describe()

,annualized_volatility
count,48.000000
mean,0.282685
std,0.064116
min,0.178616
25%,0.249486
50%,0.269649
75%,0.303184
max,0.496517
